# Tutorial 14: Reinforcement Learning (Part I)

In this tutorial, we will use two separate Reinforcement Learning approaches to perform a model-free pendulum swing-up in real time.

**Pre-requisites**

Knowledge of the fundamentals of RL, as well as Dynamic Programming.

**Goals**

Demonstrating the properties of DDPG and SAC. Showcasing the capabilities of CloudPendulum Gym.

This notebook is organized as follows:

    1. Recap: Reinforcement Learning methods and their classification
    2. Deep Deterministic Policy Gradient
    3. Soft Actor-Critic
        3.1. SAC training in simulation
        3.2. Training on cloudpendulum Gym


# 1. Reinforcement Learning recap

Fundamentally, Reinforcement Learning consists in training agents using trial and error.
The **agent** interacts with an **environment** by performing **actions**, which modify the **state** of the environment. As the agent interacts with the environment, it perceives **observations** and is given **rewards**, which guide its decisions. This happens because the goal of the agent is to maximize the rewards it accumulates. The cummulative reward receives the name **return**.

As the agent learns, its **policy** will take shape. The role of the policy is to determine the actions of the agent depending on its observations. The central problem in Reinforcement Learning can be written as:

$$\pi^* = \text{argmax}_\pi E \left[ \sum^T_{t=0}r_t(s_t,a_t) \right]$$

where $\pi$ is the policy and $\pi^*$ denotes the optimal policy. $E \left[ \sum^T_{t=0}r_t(s_t,a_t) \right]$ is the expected cummulative reward during a trajectory, with $r_t$ being the reward at time $t$ given state $s_t$ and action $a_t$. This may remind you of Bellman's equation:

$$V^*(s_t) = \max_a E \left[r(s_t, a_t) + \gamma V^*(s_{t+1}) \right]$$

In this version, the optimal cost-to-go (in this case, the reward) at one state, $V^*(s_t)$ is the maximum return you can expect if you find yourself at $s_t$ during a trajectory. Using Bellman's optimality principle we also reach the Q-function, which defines the optimal state-action pair as the one that provides the highest expected return out of all the feasible actions at a given state.

$$Q^*(s_t,a_t) = E \left[r(s_t, a_t) + \gamma \max_{a_{t+1}}Q^*(s_{t+1},a_{t+1}) \right] $$

### Types of RL algorithms

A very common way to classify RL algorithms is by putting them in the model-based or model-free categories. In this tutorial, we will cover two model-free approaches, Deep Deterministic Policy Gradient (DDPG) and Soft Actor-Critic (SAC)

<div style="display: flex; justify-content: space-around;">
    <div><img src="media/rl_algorithms_9_15.svg" width="800"></div>
</div>
Source: https://spinningup.openai.com

Another way in which we differentiate between algorithms is by dividing them between on- and off-policy. On-policy algorithms refine the policy by testing it and learning from the data that the policy generates, whereas off-policy algorithms use pre-existing data. One of the advantages of off-policy algorithms is that they can re-use training data.

# 2. Deep Deterministic Policy Gradient

The first method we will look at today is DDPG. The fundamental idea behind it is to have our algorithm learn the Q-function, as well as the optimal policy at the same time. This method is designed to deal exclusively with continuous action spaces. It deals with those by assuming that $Q^*$ is differentiable with respect to $a$. This is a Deep Learning method because it uses neural networks to approximate the Q-function and policy.

These two networks are called actor for the network that learns the policy; and critic for the network that learns the Q-function. Additionally, we keep a weighted average of both networks around as time goes on, which we call target actor and target critic networks, respectively. The goal of the critic network is to minimize $L$, the expected difference mean-squared Bellman error.

  $$ L(\phi, \mathcal{D}) = {E}_{(s,a,r,s') \sim \mathcal{D}}\left[(Q_\phi(s,a) - y)^2\right] $$

  where $y = r+ \gamma(1-d) \max_{a_{t+1}} Q_\phi(s_{t+1}, a_{t+1})$

Before starting our DDPG loop, we need to define a few variables:

- The reward function $r(s,a)$, which we want our policy to maximize.

- A set of transitions $\mathcal{D}$. $\mathcal{D}$ stores the states, actions, rewards, next states, and $d$, which indicates whether the final state is reached.

- Some tuneable parameters like the learning rate $\eta$ or the target network update rate $\rho$.

The typical DDPG training loop works as follows:

- The agent observes the state, computes the action, and executes it.

  $$a = \mu_\theta(s) + \mathcal{N}$$

  where $\mu_\theta$ is the actor network with parameters $\mu$ and $\mathcal{N}$ is an added random noise. Since the policy is deterministic, this allows us to explore a wider range of actions instead of always ascending the gradient in the exact same way.

- Observe the reward, next state, and whether the $s_{t+1}$ is terminal. Store this data in $\mathcal{D}$. If the next state is terminal, reset the state.

- Sample a batch of transitions, $B$ from $\mathcal{D}$ and compute the critic. When we update the critic, we use the target networks for stability.

  $$ y(r,s_{t+1},d) = r+ \gamma(1-d) \max_{a_{t+1}} Q_{\phi_{targ}}(s_{t+1}, \mu_{\theta_{targ}}(s_{t+1}))$$

- Update the critic network.

  $$ \phi = \phi -\eta \nabla_\phi \frac{1}{\left| B \right|} \sum_{(s,a,r,s_{t+1},d)\in B} (Q_\phi(s,a)-y(r,s_{t+1},d))^2$$

- Update the actor network.

  $$ \theta = \theta +\eta \nabla_\theta \frac{1}{\left| B \right|} \sum_{s\in B} Q_\phi(s,\mu_\theta(s)) $$

- Update the target networks.

  $$ \phi_{targ} = \rho \phi_{targ} + (1-\rho)\phi$$

  $$ \theta_{targ} = \rho \theta_{targ} + (1-\rho)\theta$$

Then, we run this loop successively until convergence.

# 3. Soft Actor-Critic

DDPG, as its name indicates, produces a deterministic policy. This means that a certain observation will always result in the same action. To mitigate this, we added some noise, which made the algorithm explore more solutions. However, this noise is hand-tuned and susceptible to performance issues depending on the environment. One of the ways to mitigate this is by using a stochastic policy instead. Deterministic policies will always provide the same action in response to a state. Stochastic policies put out a probability distribution as a function of the observed state.

When it comes to SAC, in addition to maximizing the return, we also seek to maximize entropy, that is, the randomness of the policy. Entropy is a measure of the randomness of a probability distribution. The more uncertain the action given a state, the higher the entropy. In mathematical terms, this updates our RL problem to:

$$\pi^* = \text{argmax}_\pi E \left[ \sum^\infty_{t=0} \gamma^t \left( R(s_t,a_t,s_{t+1}) + \alpha H(\pi(\cdot \mid s_t)) \right) \right]$$

where $R$ is the reward received for taking action $a_t$ at state $s_t$ and it resulting in a transition to state $s_{t+1}$. $H(\pi(\cdot \mid s_t)$ is the entropy associated to the policy at state $s_t$, which is calculated as $H(\pi(\cdot \mid s_t) = E \left[ -\log \pi(a \mid s_t) \right]$. $\alpha$ is a temperature parameter controlling the trade-off between maximizing the reward or the entropy. 

The Q-function and cost-to-go functions are also updated to include an entropy-maximizing term.

$$ Q^\pi(s,a) = E \left[ \sum^\infty_{t=0} \gamma^t R(s_t,a_t,s_{t+1} + \alpha \sum^\infty_{t=0} \gamma^t H(\pi(\cdot \mid s_t))\right]$$

$$ V^\pi (s_t) = E \left[ Q^(s_t,a_t)\right] + \alpha H(\pi(\cdot \mid s_t))$$

The resulting Bellman equation is then:

$$Q^\pi(s_t, a_t) = E\left[R(s_t, a_t, s_{t+1}) + \gamma\left(Q^\pi(s_{t+1}, a_{t+1}) + \alpha H (\pi(\cdot \mid s_{t+1}))\right)\right]$$

The typical SAC training loop is similar to DDPG, with three key differences: the policy is stochastic, two critic networks are used instead of one, and an entropy term is added to the objective. The loop works as follows:

- The agent observes the state and samples an action directly from the policy, without added noise:
  $$\tilde{a} \sim \pi_\theta(\cdot \mid s)$$
  Exploration is no longer a concern here, as the stochastic policy naturally explores by sampling different actions in the same state.
  
- Observe the reward, next state, and whether $s_{t+1}$ is terminal. Store $(s, a, r, s_{t+1}, d)$ in $\mathcal{D}$. If the next state is terminal, reset the state.


- Sample a batch of transitions $B$ from $\mathcal{D}$ and compute the target for both critics, using the minimum of the two target critics to reduce overestimation bias:
$$y(r, s_{t+1}, d) = r + \gamma(1-d)\left(\min_{i=1,2} Q_{\phi_{\text{targ},i}}(s_{t+1}, \tilde{a}_{t+1}) - \alpha \log \pi_\theta(\tilde{a}_{t+1} \mid s_{t+1})\right), \quad \tilde{a}_{t+1} \sim \pi_\theta(\cdot \mid s_{t+1})$$

- Update both critic networks independently against the shared target $y$.
$$\phi_i = \phi_i - \eta \nabla_{\phi_i} \frac{1}{|B|} \sum_{(s,a,r,s_{t+1},d) \in B} (Q_{\phi_i}(s,a) - y(r,s_{t+1},d))^2 \quad \text{for } i=1,2$$
- Update the actor network.
 $$\theta = \theta + \eta \nabla_\theta \frac{1}{|B|} \sum_{s \in B} \left(\min_{i=1,2} Q_{\phi_i}(s, \tilde{a}_\theta(s)) - \alpha \log \pi_\theta(\tilde{a}_\theta(s) \mid s)\right)$$

- Update the target networks, as in DDPG:
 $$\phi_{\text{targ},i} = \rho \phi_{\text{targ},i} + (1-\rho)\phi_i \quad \text{for } i=1,2$$
 Note that SAC has no target actor — the current actor $\pi_\theta$ is used directly when computing targets.

Then, we run this loop successively until convergence.

Now it's time to put SAC to the test. Before the next section, we will need to gather the necessary imports. If you want to use the hardware on the cloud for training and testing, make sure to add your user token too!

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib
import random 
import time

from IPython.display import HTML

from pendulum_plant.pendulum_plant import PendulumPlant
from pendulum_plant.simulation import Simulator

from sac.sac_parallel_sequential import sac_trainer
from sac.sac_controller import SacController

from pendulum_plant_new.misc import download_video
from IPython.display import Video

with open('token.txt', 'r') as file:
    USER_TOKEN = file.read().rstrip()

Set parameters for pendulum and enviroment.

In [ ]:
# pendulum parameters
mass = 0.06
length = 0.1
damping = 0.0004
torque_limit = 0.02
coulomb_fric = 0.0
inertia = mass*length**2
gravity = 9.81

# environment parameters
dt = 0.02
max_steps = 500 
reward_type = "combined_reward"
target = [np.pi, 0]
target_epsilon = [0.05, 0.05]
random_init = "everywhere"
integrator = "runge_kutta"


## 3.1. SAC training in simulation

Since we already have a simulated environment from previous tutorials, we can use it to train our policy. Using simulations to train agents is often cheaper and safer than doing it on the real thing. Additionally, it can greatly speed up the process of training, assuming that we can simulate faster than real time.

In [ ]:
# training parameters
log_dir = "log_data/sac_parallel_training"
base_log_dir = "log_data/sac_parallel_evaluation_gs_4000"
learning_rate = 0.0003
n_episodes=50
eval_every = 4
eval_episodes = 1
start_training = 2000 #changed from 2000
number_of_envs = 4
gradient_steps=1000 # changed from 1000
batch_size=256 # changed from 256
reward_limit=21_000
device = 'cuda'

ec_times = []
pu_times = []

def run_evaluation():
    print(f"Running evaluation run 0")
    log_dir = os.path.join(base_log_dir, f"run_0")
    trainer = sac_trainer(log_dir=log_dir)
    trainer.init_pendulum(mass=mass,
                        length=length,
                        inertia=inertia,
                        damping=damping,
                        coulomb_friction=coulomb_fric,
                        gravity=gravity,
                        torque_limit=torque_limit)

    trainer.init_environment(dt=dt,
                            integrator=integrator,
                            max_steps=max_steps,
                            reward_type=reward_type,
                            target=target, 
                            state_target_epsilon=target_epsilon,
                            random_init=random_init,
                            state_representation=3,
                            n_envs=number_of_envs)

    trainer.init_agent(learning_rate=learning_rate,
                    warm_start=False,
                    warm_start_path="",
                    device=device,
                    verbose=1)

    print(f"Training using {device}")
    start_time = time.time()
    trainer.train(n_episodes=n_episodes,
                gradient_steps=gradient_steps,
                batch_size=batch_size,
                eval_every=eval_every,
                eval_episodes=eval_episodes,
                start_training=start_training,
                save_path=log_dir,
                reward_limit=reward_limit)
    print(f"Training total time  with {device}: {time.time()-start_time}")


# ------ n env evaluation --------
base_log_dir = "log_data/sac"
n_episodes=67
number_of_envs = 4 # changed from 4
run_evaluation()

Now that we have hopefully successfully trained our agent, it's time to test it in a simulation.

In [ ]:
pendulum = PendulumPlant(mass=mass,
                         length=length,
                         damping=damping,
                         gravity=gravity,
                         coulomb_fric=coulomb_fric,
                         inertia=inertia,
                         torque_limit=torque_limit)

sim = Simulator(plant=pendulum)

model_path = "log_data/sac/run_0/best_model.zip" #"log_data/sac_parallel_evaluation/run_0/best_model.zip"

controller = SacController(model_path=model_path,
                           torque_limit=torque_limit,
                           use_symmetry=False,
                           state_representation=3,
                           deterministic=True)

# simulate
x0_sim = [0.0, 0.0]
dt = 0.02
t_final = 10
integrator = "runge_kutta"

T, X, U = sim.simulate(t0=0.0,
                                x0=x0_sim,
                                tf=t_final,
                                dt=dt,
                                controller=controller,
                                integrator=integrator)


fig, ax = plt.subplots(3, 1, figsize=(18, 6), sharex="all")
print(f"Final torque: {U[-1]}")
ax[0].plot(T, np.asarray(X).T[0], label="theta")
ax[0].hlines(y=[-np.pi, np.pi], xmin=0, xmax = t_final, linestyles = ':', colors='g')
ax[0].set_ylabel("angle [rad]")
ax[0].legend(loc="best")
ax[1].plot(T, np.asarray(X).T[1], label="theta dot")
ax[1].set_ylabel("angular velocity [rad/s]")
ax[1].legend(loc="best")
ax[2].plot(T, np.asarray(U).flatten(), label="u")
ax[2].set_xlabel("time [s]")
ax[2].set_ylabel("input torque [Nm]")
ax[2].hlines(y=0, xmin=0, xmax=t_final, linestyles = ':', colors='g')
ax[2].legend(loc="best")
plt.show()


If all went well, you should see the pendulum gathering speed and reaching an upright position then stopping. If this is a bit difficult to visualize, we also have an animation showing the simulation.

In [ ]:
from pendulum_plant_new.pendulum_plant import PendulumPlant, plot_timeseries

pendulum2 = PendulumPlant(mass=mass,
                        length=length,
                        damping=damping,
                        gravity=gravity,
                        inertia=inertia,
                        torque_limit=torque_limit)

Tsim, Xsim, Usim, anim = pendulum2.simulate_and_animate(
              t0=0.0,
              x0=[0.0, 0.0],
              tf=5.0,
              dt=0.002,
              controller=controller,
              integrator="runge_kutta", anim_dt = 0.04)

HTML(anim.to_html5_video())
plot_timeseries(Tsim, Xsim, Usim)

Once we have verified that the policy works in simulation, it is time to test it on the real pendulum. Run the cell below to make sure you have the adequate permissions and a valid token.

In [ ]:
from cloudpendulumclient.client import Client
client = Client()
client.get_user_info(USER_TOKEN) # Monitor the status of your user token

And run the next cell to connect to a pendulum cell on the cloud.

In [ ]:
tf = 10 # Final time (s)
dt = 0.01 # Time step (s)
Treal, Xreal, Ureal, Ureal_des, url, path = pendulum2.run_on_hardware(user_token=USER_TOKEN,
                                                                            tf=5.0,
                                                                            dt=0.01,  
                                                                            controller=controller) 

# Measured Position
plt.figure
plt.plot(Treal, np.asarray(Xreal).T[0])
plt.xlabel("Time (s)")
plt.ylabel("Position (rad)")
plt.title("Position (rad) vs Time (s)")
plt.show()

# Measured Velocity
plt.figure
plt.plot(Treal, np.asarray(Xreal).T[1])
plt.xlabel("Time (s)")
plt.ylabel("Velocity (rad/s)")
plt.title("Velocity (rad/s) vs Time (s)")
plt.show()

# Measured Torque
plt.figure
plt.plot(Treal, Ureal)
plt.xlabel("Time (s)")
plt.ylabel("Torque (Nm)")
plt.title("Torque (Nm) vs Time (s)")
plt.show()

video_path = download_video(url)
from IPython.display import Video
Video(path)

## Think-Pair-Share

Did the sim-to-real transfer work? Why/Why not? What are the implications when it comes to the robustness of this type of controllers? How do you think SAC's entropy maximization plays into this?

We can also run the next cell to visualize the value function.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def visualize_policy_with_stabilization_view(controller, 
                                             theta_range=(-np.pi, np.pi), 
                                             vel_range=(-8, 8), 
                                             stab_range=0.6,
                                             bins=200,
                                             zoom_bins=100):
    """
    Plots two heatmaps:
    1. Full (theta, velocity) policy
    2. Zoomed view around stabilization point ±π, shown as deviation from upright

    Parameters
    ----------
    controller : sac_controller
        The SAC controller.
    theta_range : tuple
        Range for theta in full plot.
    vel_range : tuple
        Range for velocity in both plots.
    stab_range : float
        Maximum deviation (in radians) from the upright position (π or -π).
    bins : int
        Resolution for full plot.
    zoom_bins : int
        Resolution for zoom plot.
    """
    def compute_actions(theta_vals, vel_vals):
        actions = np.zeros((len(vel_vals), len(theta_vals)))
        for i, th in enumerate(theta_vals):
            for j, vel in enumerate(vel_vals):
                action = controller.get_control_output(meas_pos=th, meas_vel=vel)
                actions[j, i] = action
        return actions

    # === Full policy ===
    thetas = np.linspace(*theta_range, bins)
    vels = np.linspace(*vel_range, bins)
    full_actions = compute_actions(thetas, vels)

    # === Stabilization view around ±π ===
    delta_theta_vals = np.linspace(-stab_range, stab_range, zoom_bins)
    vels_zoom = np.linspace(-4, 4, zoom_bins)

    # Map delta_theta to wrapped angles near ±π
    theta_left = -np.pi + delta_theta_vals  # around -π
    theta_right = np.pi - delta_theta_vals  # around +π

    # Merge both into one batch
    theta_zoom = np.concatenate([theta_left, theta_right])
    delta_zoom = np.concatenate([delta_theta_vals, delta_theta_vals])  # for x-axis
    vels_zoom_full = np.tile(vels_zoom, 2)

    actions_zoom = np.zeros((zoom_bins, 2 * zoom_bins))
    for i, dt in enumerate(delta_theta_vals):
        for j, vel in enumerate(vels_zoom):
            a_left = controller.get_control_output(meas_pos=-np.pi + dt, meas_vel=vel)
            a_right = controller.get_control_output(meas_pos=np.pi + dt, meas_vel=vel)
            actions_zoom[j, i] = a_left
            actions_zoom[j, i + zoom_bins] = a_right

    # === Plotting ===
    fig, axs = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    # Full plot
    im1 = axs[0].imshow(full_actions, extent=[*theta_range, *vel_range],
                        origin='lower', aspect='auto', cmap='coolwarm')
    axs[0].set_title("Full Policy")
    axs[0].set_xlabel("Theta (rad)")
    axs[0].set_ylabel("Angular Velocity")
    fig.colorbar(im1, ax=axs[0], label='Torque')

    # Stabilization plot
    x_extent = [-stab_range, stab_range]
    im2 = axs[1].imshow(actions_zoom, extent=[x_extent[0], x_extent[1], -4, 4],
                        origin='lower', aspect='auto', cmap='coolwarm')
    axs[1].set_title("Stabilization Near ±π")
    axs[1].set_xlabel("ΔTheta from ±π (rad)")
    fig.colorbar(im2, ax=axs[1], label='Torque')

    plt.suptitle("SAC Policy: Full and Stabilization Region Views")
    plt.tight_layout()
    plt.show()
visualize_policy_with_stabilization_view(controller)

Does this remind you of anything?

## 3.2. Training on cloudpendulum gym

Training on a simulator can be convenient, but it is not always feasible. Some robots may be complex and thus diffcult to model. Even if we do model them, the sim-to-real gap can be problematic. As we have seen before with other control strategies, there are always unmodelled effects affecting the dynamics of our systems. The more complex or unstable a system is, the more robust its controllers will need to be. By working directly on the hardware, we do not need to worry about the sim-to-real gap.

Run the next cell to get the necessary imports and adjust the configuration of the SAC algorithm on CloudPendulum Gym.

In [ ]:
from sac.sac_parallel_disconnected_hw import sac_trainer
from sac.sac_controller import SacController

# environment parameters
dt = 0.02
max_steps = 500 
reward_type = "combined_reward"
target = [np.pi, 0]
target_epsilon = [0.1, 0.1]
random_init = "everywhere"
random_init_eval = "False"
dt_step_scaling=0.80

Next, run the following cell to start training. Every group should only use 1 environment.

In [ ]:
# training parameters
log_dir = "log_data/sac_hw/run_1"
learning_rate = 0.0003
n_episodes=200
gradient_steps=1000
batch_size=1024
eval_every = 5
eval_episodes = 1
start_training = 2000
number_of_envs = 1 # Do not modify!
reward_limit = 5_000

trainer = sac_trainer(log_dir=log_dir, verbose=0)

trainer.init_environment(user_token=USER_TOKEN,
                         dt=dt,
                         max_steps=max_steps,
                         reward_type=reward_type,
                         state_representation=3,
                         target=target, 
                         state_target_epsilon=target_epsilon,
                         random_init=random_init,
                         random_init_eval=random_init_eval,
                         dt_step_scaling=dt_step_scaling)

trainer.init_agent(learning_rate=learning_rate,
                   warm_start=False,
                   )

trainer.train(n_episodes=n_episodes,
              gradient_steps=gradient_steps,
              batch_size=batch_size,
              eval_every=eval_every,
              eval_episodes=eval_episodes,
              start_training=start_training,
              save_path=log_dir,
              number_of_envs=number_of_envs,
              reward_limit=reward_limit)



Once the training has finished, you can optionally run the next cell to get a picture of how the training evolved.

In [ ]:
# Create a video showing training progress

from sac.sac_analysis_utils import create_evaluation_progress_video

folder_path = "log_data/sac_hw/run_1/evaluation_videos"
create_evaluation_progress_video(folder_path)

Lastly, run the next cell to run an experiment and check that your controller works.

In [ ]:
controller_real = SacController(model_path=model_path,
                           torque_limit=torque_limit,
                           use_symmetry=False,
                           state_representation=3,
                           deterministic=False)

tf = 10 # Final time (s)
dt = 0.01 # Time step (s)
Treal2, Xreal2, Ureal2, Ureal2_des, url2, path2 = pendulum2.run_on_hardware(user_token=USER_TOKEN,
                                                                            tf=5.0,
                                                                            dt=0.01,  
                                                                            controller=controller_real) 

# Measured Position
plt.figure
plt.plot(Treal, np.asarray(Xreal2).T[0])
plt.xlabel("Time (s)")
plt.ylabel("Position (rad)")
plt.title("Position (rad) vs Time (s)")
plt.show()

# Measured Velocity
plt.figure
plt.plot(Treal2, np.asarray(Xreal2).T[1])
plt.xlabel("Time (s)")
plt.ylabel("Velocity (rad/s)")
plt.title("Velocity (rad/s) vs Time (s)")
plt.show()

# Measured Torque
plt.figure
plt.plot(Treal2, Ureal2)
plt.xlabel("Time (s)")
plt.ylabel("Torque (Nm)")
plt.title("Torque (Nm) vs Time (s)")
plt.show()

video_path = download_video(url2)
from IPython.display import Video
Video(path2)

## Think-Pair-Share

Do you see any differences with respect to the policy trained on the simulation? If any of your policies turned out unsatisfactory, revise the settings and run the optimization again. 